## Cell 1 — Library Imports & Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import warnings
import joblib
import os
warnings.filterwarnings("ignore")

from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from scipy.fft import fft, fftfreq

from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay, RocCurveDisplay, precision_recall_curve, average_precision_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
import sklearn

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print('imblearn not found. Install with: pip install imbalanced-learn')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

# GLOBAL STYLING
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
}),
PALETTE = {'Control': '#2ecc71', 'Cancer': '#e74c3c', 'Benign': '#e67e22'}
BINARY_PALETTE = {0: '#2ecc71', 1: '#e74c3c'}

# CONSTANTS
RANDOM_STATE = 42
N_FOLDS = 5
SAMPLING_RATE = 100
MEASUREMENT_DURATION = 60
N_POINTS = SAMPLING_RATE * MEASUREMENT_DURATION
N_SENSORS = 22
N_MOS = 10
N_EC = 8
N_PID = 1
N_QCM = 2

print('Libraries loaded')
print(f'Constants initialized: {N_FOLDS} folds, {N_POINTS} points')
print(f'scikit-learn: {sklearn.__version__}, imbalanced-learn: {"available" if SMOTE_AVAILABLE else "not available"}')
print(f'numpy: {np.__version__}, pandas: {pd.__version__}')


Note: you may need to restart the kernel to use updated packages.
Libraries loaded
Constants initialized: 5 folds, 6000 points
scikit-learn: 1.8.0, imbalanced-learn: available
numpy: 1.26.4, pandas: 3.0.2


## Cell 2 — Raw Sensor Calibration Functions

In [1]:
# ══════════════════════════════════════════════════════
# 2A. MOS SENSORS — Power-Law Model
# Rs/R0 = a · C^(-b)  →  C = (Rs/R0 / a)^(-1/b)

def mos_model(C, a, b):
    return a & np.power(C, -b)

def calibrate_mos(known_ppb, measured_Rs_R0):
    popt, pcov = curve_fit(
        mos_model, known_ppb, measured_Rs_R0,
        p0=[1.0, 0.3], bounds=([0, 0], [np.inf, np.inf]),
        maxfev=10000
    )
    a, b = popt
    perr = np.sqrt(np.diag(pcov))
    print(f'  MOS fit → a={a:.4f}±{perr[0]:.4f}, b={b:.4f}±{perr[1]:.4f}')
    return {'a': a, 'b': b, 'type': 'MOS'}

def mos_to_ppb(Rs_R0, params):
    a, b = params['a'], params['b']
    Rs_R0_clamped = np.clip(Rs_R0, 1e-6, None)
    return np.power(Rs_R0_clamped / a, -1.0 / b)

# ══════════════════════════════════════════════════════
# 2B. EC SENSORS — Linear Model
#     C (ppb) = (I - I_zero) / sensitivity

def calibrate_ec(known_ppb, measured_nA):
    lr = LinearRegression().fit(
        np.array(known_ppb).reshape(-1, 1),
        np.array(measured_nA)
    )
    sensitivity = lr.coef_[0]
    I_zero = lr.intercept_
    print(f'  EC fit  → sensitivity={sensitivity:.4f} nA/ppb, I_zero={I_zero:.2f} nA')
    return {'sensitivity': sensitivity, 'I_zero': I_zero, 'type': 'EC'}

def ec_to_ppb(I_nA, params):
    conc = (np.array(I_nA) - params['I_zero']) / params['sensitivity']
    return np.clip(conc, 0, None)

# ══════════════════════════════════════════════════════
# 2C. PID SENSOR — Linear + Response Factor Correction
#     C (ppb) = V_output / (RF_compound × Sensitivity_ref)

PID_RESPONSE_FACTORS = {
    'benzene':      0.53,
    'toluene':      0.54,
    'ethylbenzene': 0.52,
    'xylene':       0.52,
    'styrene':      0.40,
    'isobutylene':  1.00,
    'generic_btex': 0.52,
}

def calibrate_pid(known_ppb_isobutylene, measured_V):
    lr = LinearRegression().fit(
        np.array(known_ppb_isobutylene).reshape(-1, 1),
        np.array(measured_V)
    )
    sensitivity = lr.coef_[0]
    V_zero = lr.intercept_
    print(f'  PID fit → sensitivity={sensitivity:.6f} V/ppb, V_zero={V_zero:.4f} V')
    return {'sensitivity': sensitivity, 'V_zero': V_zero, 'type': 'PID'}

def pid_to_ppb(V_raw, params, compound='generic_btex'):
    RF = PID_RESPONSE_FACTORS.get(compound, 0.52)
    V_corrected = np.array(V_raw) - params['V_zero']
    conc = V_corrected / (RF * params['sensitivity'])
    return np.clip(conc, 0, None)

# ══════════════════════════════════════════════════════
# 2D. QCM SENSORS — Sauerbrey Equation
#     Δm = -Δf · A·√(ρq·μq) / (2f₀²)
#     C (ppb) via ideal gas law

QCM_CONSTANTS = {
    'f0':      10e6,     # Hz, resonant frequency
    'A':       1.54e-5,  # m², electrode area
    'rho_q':   2650,     # kg/m³, quartz density
    'mu_q':    2.947e10, # Pa, quartz shear modulus
    'V_cell':  50e-6,    # m³, sensor cell volume
    'T_K':     298,      # K, temperature (25°C)
    'P_Pa':    101325,   # Pa, atmospheric pressure
    'R_gas':   8.314,    # J/(mol·K)
}

VOC_MW = {
    'acetone':      0.05808,
    'isoprene':     0.06811,
    'ethanol':      0.04607,
    'pentane':      0.07215,
    'generic':      0.06000,
}

def qcm_to_ppb(delta_f, compound='generic', constants=None):
    if constants is None:
        constants = QCM_CONSTANTS
    
    f0    = constants['f0']
    A     = constants['A']
    rho_q = constants['rho_q']
    mu_q  = constants['mu_q']
    V_cell = constants['V_cell']
    T_K   = constants['T_K']
    P_Pa  = constants['P_Pa']
    R_gas = constants['R_gas']
    MW    = VOC_MW.get(compound, VOC_MW['generic'])
    
    # Sauerbrey: Mass-Change
    delta_m = (-np.array(delta_f) * A * np.sqrt(rho_q * mu_q)) / (2 * f0**2)
    delta_m = np.clip(delta_m, 0, None)  # mass can only increase on adsorption
    
    # Moles Adsorbed
    moles = delta_m / MW
    
    # Gas-phase concentration (mol/m³) → molar fraction → ppb
    # Using ideal gas: n/V = P/(RT) for total gas
    total_molar_density = P_Pa / (R_gas * T_K)  # mol/m³
    molar_fraction = (moles / V_cell) / total_molar_density
    ppb = molar_fraction * 1e9
    
    return ppb

# ══════════════════════════════════════════════════════
# 2E. Temperature & Humidity Compensation

def compensate_TRH(C_raw, T_C, RH_pct,
                    T_ref=25.0, RH_ref=50.0,
                    alpha_T=-0.008, alpha_RH=-0.003):
    delta_T  = np.array(T_C) - T_ref
    delta_RH = np.array(RH_pct) - RH_ref
    correction_factor = 1 + alpha_T * delta_T + alpha_RH * delta_RH
    correction_factor = np.clip(correction_factor, 0.5, 1.5)  # safety bounds
    return np.array(C_raw) / correction_factor

print('Calibration functions:')
print('   ├─ MOS  : power-law model (Rs/R0 = a·C^-b)')
print('   ├─ EC   : linear model (I = S·C + I_zero)')
print('   ├─ PID  : linear + response factor correction')
print('   ├─ QCM  : Sauerbrey equation + ideal gas law')
print('   └─ T/RH : linear compensation surface')


Calibration functions:
   ├─ MOS  : power-law model (Rs/R0 = a·C^-b)
   ├─ EC   : linear model (I = S·C + I_zero)
   ├─ PID  : linear + response factor correction
   ├─ QCM  : Sauerbrey equation + ideal gas law
   └─ T/RH : linear compensation surface
